# ResNet18 on manually labeled ERP image patterns

This notebook syncs local Label Studio annotations, builds fixed-trial augmented ERP images from the labeled sources, and runs a 5-fold cross-validation with an ImageNet-pretrained ResNet18. The prediction target remains binary: `class` versus `no_class`.

Implementation details are in `resnet18_labeled_erp_cv.jl` so the experiment can also be run from the terminal. Preprocessing is `sort -> zscore_timepoints -> Gaussian smoothing -> resize 64x64`, using the shared `gaussian_reference` utility pipeline.

In [1]:
import Pkg

function find_repo_root(start::AbstractString = pwd())
    candidate = start
    for _ in 1:8
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
        candidate = dirname(candidate)
    end
    error("Could not locate repo root from ", start)
end

REPO_ROOT = find_repo_root()
WEEK21 = joinpath(REPO_ROOT, "notebooks", "week_21")
OUTPUT_DIR = joinpath(WEEK21, "outputs", "resnet18_labeled_erp_cv")

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))
using CSV, DataFrames, JSON3
REPO_ROOT

  Activating project at `~/Dokumente/BA2/notebooks/model_test`


"/home/benjamin/Dokumente/BA2"

## 1. Sync Label Studio labels

This step reads the local Label Studio SQLite DB and updates the tracking CSVs, including the most recent follow-up projects.

In [2]:
run(Cmd(`python3 $(joinpath(WEEK21, "update_labelstudio_annotation_tracking.py"))`; dir = REPO_ROOT))

Wrote /home/benjamin/Dokumente/BA2/notebooks/week_21/labelstudio_annotations_all.csv (2879 annotated tasks)
Wrote /home/benjamin/Dokumente/BA2/notebooks/week_21/labelstudio_positive_sort_variables.csv
Updated tracking CSVs under: /home/benjamin/Dokumente/BA2/notebooks/week_21/labelstudio_export_test, /home/benjamin/Dokumente/BA2/notebooks/week_21/labelstudio_export_model_prioritized_200, /home/benjamin/Dokumente/BA2/notebooks/week_21/labelstudio_export_pattern_positive_followup_1000


Process(setenv(`python3 /home/benjamin/Dokumente/BA2/notebooks/week_21/update_labelstudio_annotation_tracking.py`; dir="/home/benjamin/Dokumente/BA2"), ProcessExited(0))

## 2. Run ResNet18 cross-validation

The Julia script uses `WEEK21_TARGET_TRIALS` as the fixed trial count for every augmented ERP image; the default is 150. Positive pattern rows keep all round-robin mod-split chunks plus a filled remainder chunk when a remainder exists; `no_class` rows are augmented the same way but only one deterministic chunk is kept for class balance. Folds are stratified by the seven manual labels, so pattern types are distributed across folds. The ERP image preprocessing uses `gaussian_reference` from the shared utils with the project-wide low-pass factor and kernel size.

In [3]:
# Optional overrides before running:
# ENV["WEEK21_RESNET18_EPOCHS"] = "8"
# ENV["WEEK21_NO_CLASS_CHUNKS_PER_ORIGIN"] = "1"
get!(ENV, "WEEK21_TARGET_TRIALS", "150")

run(Cmd(`julia --project=notebooks/model_test $(joinpath(WEEK21, "resnet18_labeled_erp_cv.jl"))`; dir = REPO_ROOT))

  Activating project at `~/Dokumente/BA2/notebooks/model_test`
Precompiling packages...
   2012.4 ms  ✓ CUDA_Runtime_jll
    963.9 ms  ✓ CUDA_Compiler_jll
  23126.3 ms  ✓ CUDA
   2674.6 ms  ✓ Atomix → AtomixCUDAExt
  4 dependencies successfully precompiled in 29 seconds. 99 already precompiled.
  1 dependency had output during precompilation:
┌ Atomix → AtomixCUDAExt
│  ┌ Warning: You are using a non-official build of Julia. This may cause issues with CUDA.jl.
│  │ Please consider using an official build from https://julialang.org/downloads/.
│  └ @ CUDA ~/.julia/packages/CUDA/FJf6p/src/initialization.jl:170
└  
┌ Warning: You are using a non-official build of Julia. This may cause issues with CUDA.jl.
│ Please consider using an official build from https://julialang.org/downloads/.
└ @ CUDA ~/.julia/packages/CUDA/FJf6p/src/initialization.jl:170
Precompiling packages...
    579.5 ms  ✓ Baselet
    355.2 ms  ✓ ADTypes
    474.8 ms  ✓ InitialValues
    225.5 ms  ✓ PrettyPrint
    230.9 ms

Loading Label Studio annotations.
Resolving source origins and trial counts for 2869 labels.
Configured fixed trial count per augmented ERP image: 150 | minimum origin trial count: 297
Materializing fixed-trial augmented ERP images.
Training and validating ResNet18 with 5-fold CV.
CUDA device: NVIDIA GeForce RTX 4070
resnet18_pretrained_labeled_erp_binary | fold 1/5 | train=4268 | val=1067
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 1/8 | loss=0.74733
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 2/8 | loss=0.33364
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 3/8 | loss=0.20504
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 4/8 | loss=0.15003
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 5/8 | loss=0.12685
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 6/8 | loss=0.08255
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 7/8 | loss=0.09351
resnet18_pretrained_labeled_erp_binary_fold1 | epoch 8/8 | loss=0.09119
resnet18_pretrained_labeled_erp

Process(setenv(`julia --project=notebooks/model_test /home/benjamin/Dokumente/BA2/notebooks/week_21/resnet18_labeled_erp_cv.jl`; dir="/home/benjamin/Dokumente/BA2"), ProcessExited(0))

## 3. Inspect outputs

In [4]:
metrics_summary = CSV.read(joinpath(OUTPUT_DIR, "metrics_summary.csv"), DataFrame)
fold_metrics = CSV.read(joinpath(OUTPUT_DIR, "fold_metrics.csv"), DataFrame)
fold_classes = CSV.read(joinpath(OUTPUT_DIR, "fold_distribution_pattern_class.csv"), DataFrame)
run_config = JSON3.read(read(joinpath(OUTPUT_DIR, "run_config.json"), String))

config_subset = (
    target_trials       = run_config.target_trials,
    target_size         = run_config.target_size,
    lowpass_sigma       = run_config.lowpass_sigma,
    lowpass_kernel_size = run_config.lowpass_kernel_size,
    n_labeled_rows_used = run_config.n_labeled_rows_used,
    n_augmented_images  = run_config.n_augmented_images,
    k_folds             = run_config.k_folds,
    nepochs             = run_config.nepochs,
)
metrics_summary, config_subset

(1×11 DataFrame
 Row │ model_name                         val_accuracy_mean  val_accuracy_std  ⋯
     │ String                             Float64            Float64           ⋯
─────┼──────────────────────────────────────────────────────────────────────────
   1 │ resnet18_pretrained_labeled_erp_…           0.866542        0.00524334  ⋯
                                                               8 columns omitted, (target_trials = 150, target_size = [64, 64], lowpass_sigma = 75, lowpass_kernel_size = [21, 21], n_labeled_rows_used = 2869, n_augmented_images = 5335, k_folds = 5, nepochs = 8))

In [5]:
first(fold_classes, 12)

Row,fold,erp_class,count
,Int64,String15,Int64
1,1,diverging_bar,109
2,2,diverging_bar,109
3,3,diverging_bar,108
4,4,diverging_bar,108
5,5,diverging_bar,108
6,1,hourglass,38
7,2,hourglass,38
8,3,hourglass,39
9,4,hourglass,38
